# Native two-link posture optimization
Retain all fourteen source stages, literal S-then-H Y measurements and MSB-first count marginals. Native qfunc composition supplies three measurement programs. Every objective draws one thousand actual samples in each basis; retain the original thirty-query budget. Native COBYLA substitutes SciPy without optimizer trajectory equivalence. Repeated final estimates remain separate noisy observations.

The Notebook adds two initial three-basis diagnostics (6000draws) outside optimization. The complete workflow uses 3000draws per objective query plus7000finalreadoutdraws; the Notebook reports both totals explicitly.


In [ ]:
import json
from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.applications.optimization.robust_posture_optimization.overview.measurements import (
    BASES,
    SHOTS,
    TARGET,
    estimate,
    sample,
)
from examples.synthesis.applications.optimization.robust_posture_optimization.overview.plots import (
    plot_trace,
)
from examples.synthesis.applications.optimization.robust_posture_optimization.overview.quantum import (
    build_program,
    distribution,
)
from examples.synthesis.applications.optimization.robust_posture_optimization.overview.workflow import (
    INITIAL_PARAMETERS,
    run_workflow,
)


In [ ]:
print(
    Path(
        "examples/synthesis/applications/optimization/robust_posture_optimization/overview/quantum.py"
    ).read_text()
)


In [ ]:
preview = build_program(INITIAL_PARAMETERS, "Y")
print("Native two-qubit Y-measurement program:", preview.report["resources"])


In [ ]:
print(preview.circuit)


In [ ]:
print("Shots per measurement:", SHOTS)
print("Objective evaluation cap:", 30)
print("Degrees of freedom:", 6)
print("Unit link lengths:", (1.0, 1.0))
print("Target position:", TARGET)


In [ ]:
print("Original pi-linear initial angles:", INITIAL_PARAMETERS)


In [ ]:
programs = {basis: build_program(INITIAL_PARAMETERS, basis) for basis in BASES}
print({basis: program.report["resources"] for basis, program in programs.items()})


In [ ]:
initial_measurements = tuple(
    sample(distribution(INITIAL_PARAMETERS, basis), basis, 1700 + index)
    for index, basis in enumerate(BASES)
)
print("Initial actual counts in 00, 01, 10, 11 order:", initial_measurements)


In [ ]:
for reading in initial_measurements:
    print(reading.basis, "Source MSB-first marginals:", reading.marginals)


In [ ]:
initial_estimate = estimate(INITIAL_PARAMETERS, seed=1800)
print("Separate finite-shot initial position:", initial_estimate.position)
print("Squared target error:", initial_estimate.cost)


In [ ]:
workflow = run_workflow()
print("Actual objective queries:", len(workflow.trace))
print("Termination:", workflow.termination, "; converged:", workflow.converged)
print("Best actually evaluated angles:", workflow.selected.parameters)
print("Selected measured cost:", workflow.selected.estimate.cost)
plot_path = plot_trace(
    workflow, Path("artifacts/development/robust-posture/cost-convergence.png")
)
display(Image(filename=str(plot_path)))


In [ ]:
print("Final independent 1,000 source Y-basis samples:", workflow.final_y)


In [ ]:
print("First independent final position estimate:", workflow.positions[0].position)


In [ ]:
print("predict_arm_position:", workflow.positions[1].position)
print("target_arm_position:", TARGET)
report = {
    "width": 2,
    "parameters": list(workflow.selected.parameters),
    "evaluation_budget": 30,
    "evaluations": len(workflow.trace),
    "shots_per_basis": SHOTS,
    "objective_shots": sum(
        sum(reading.counts)
        for row in workflow.trace
        for reading in row.estimate.measurements
    ),
    "diagnostic_draws": 6000,
    "final_readout_draws": 7000,
    "workflow_draws": 3000 * len(workflow.trace) + 7000,
    "notebook_draws": 3000 * len(workflow.trace) + 13000,
    "final_y_counts": list(workflow.final_y.counts),
    "positions": [list(value.position) for value in workflow.positions],
    "target": list(TARGET),
    "trace": [value.estimate.cost for value in workflow.trace],
    "selected_cost": workflow.selected.estimate.cost,
    "termination": workflow.termination,
    "converged": workflow.converged,
}
print("ARTIFACT_JSON=" + json.dumps(report))
